# Random rollout on CartPole

This notebook shows the smallest end-to-end loop: build a **vector** environment from `EnvConfig`, sample random actions, and inspect the TensorDict records returned each step.

See [docs/guide.md](../docs/guide.md) for the full step contract.

## Imports

`EnvConfig` holds preset constructors (`cartpole`, `frozenlake`, `atari`, …). `make_vector_env` wraps Gymnasium and formats steps as `(data, metadata, metrics)`.

In [ ]:
from mouse.envs import EnvConfig, make_vector_env

## Configure the environment

- `num_envs=4` runs four CartPole instances in parallel.
- `max_episode_steps` caps episode length (truncation).
- `seed` fixes RNG for reproducibility.

In [ ]:
cfg = EnvConfig.cartpole(seed=0, num_envs=4, max_episode_steps=500)
env = make_vector_env(cfg)

## Rollout loop

Each `env.step` returns:

- **`data`**: one `TensorDict` per parallel env (observation, reward, `done`, `time`, …).
- **`metadata`**: batch fields such as `group_ids` and `episode_index`.
- **`metrics`**: episode-level stats (e.g. cumulative reward when an episode ends).

The **first** `step` call performs an internal reset (actions are ignored); see the guide for the initial frame (`time == 0`).

In [ ]:
for step in range(1000):
    actions = env.sample_random_actions()
    data, metadata, metrics = env.step(actions)

    if step % 100 == 0:
        group_ids = metadata["group_ids"]
        for i, td in enumerate(data):
            print(
                f"step={step:4d}  "
                f"group_id={group_ids[i]}  "
                f"time={td['time'].item()}  "
                f"done={td['done'].item()}  "
                f"reward_step={td['reward']['step'].item():.3f}  "
                f"reward_episodic={td['reward']['episodic'].item():.3f}"
            )

## Cleanup

In [ ]:
env.close()